## **Execução de RPC + Ollama no Google Colab**  

Este tutorial apresenta um guia detalhado para a configuração e execução de **RPC + Ollama** em um ambiente **Google Colab**.  

### **Requisitos**  

Antes de iniciar, certifique-se de atender aos seguintes requisitos:  

- Conta no **Ngrok** com um **token de autenticação** válido.  
- Acesso a um ambiente **Google Colab** com suporte a **GPU Nvidia (T4 - free tier)**.  
- **Python** instalado na máquina local.  

Os próximos passos detalham o processo de instalação e configuração, garantindo a correta execução do ambiente para testes e experimentação.



---



**I. Instalação das Dependências do Ollama**


> O pacote `pciutils` é necessário para que o Ollama consiga identificar corretamente o tipo de GPU disponível na instância do Colab.

> A instalação do Ollama na instância em tempo de execução será realizada pelo seguinte comando `sh curl -fsSL https://ollama.com/install.sh | sh`

In [ ]:
!sudo apt update
!sudo apt install -y pciutils
!curl -fsSL https://ollama.com/install.sh | sh

**II. Instalação das Dependências do Python**  

> O pacote `langchain-ollama` é necessário para integrar o Ollama com a biblioteca **Langchain**, facilitando a interação com modelos de linguagem.  

> O pacote `pyngrok` é necessário para configurar e gerenciar o túnel do Ngrok diretamente a partir do código Python.  

In [ ]:
!pip install langchain-ollama
!pip install pyngrok

**III. Autenticar Ngrok com Authtoken**
> Esta parte é essencial para comunição fora do Colab, para obter o token basta acessar a página `https://dashboard.ngrok.com/get-started/your-authtoken`, e copiar o token após se autenticar.


In [ ]:
!ngrok authtoken COLOCAR_SEU_TOKEN_AQUI

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


**IV. Execução do Ollama**  

> Para utilizar o Ollama, é necessário que ele seja executado como um serviço em segundo plano, paralelo aos seus scripts. No entanto, como os Jupyter Notebooks são projetados para rodar os blocos de código de forma sequencial, isso dificulta a execução simultânea de dois blocos de código. Como solução, vamos criar um serviço utilizando o módulo `subprocess` em Python, garantindo que a execução de uma célula não bloqueie a execução das demais.

> Além disso, para garantir que o serviço Ollama esteja completamente em funcionamento antes de baixar o modelo, utilizamos um pequeno delay com o comando `time.sleep(5)`, o que dá tempo para o serviço ser inicializado corretamente.

In [ ]:
import threading
import subprocess
import time

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

**V. Baixando o Modelo**  

> Para utilizar o modelo LLM, é necessário fazer o download utilizando o comando `ollama pull llama3.1`. Este comando irá baixar o modelo Llama 3.1 8b, que pode ser utilizado em seu ambiente Colab.  

> Se desejar utilizar outros modelos, você pode consultar a lista completa de modelos disponíveis no site oficial do Ollama: [https://ollama.com/library](https://ollama.com/library).  

In [ ]:
!ollama pull llama3.1

**VI. Iniciar servidor RPC com Ollama e RPC**
> Este código configura um servidor XML-RPC que permite interagir com o modelo de texto rodando no Ollama + Langchain.



In [ ]:
from xmlrpc.server import SimpleXMLRPCServer
from xmlrpc.server import SimpleXMLRPCRequestHandler
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama.llms import OllamaLLM
from pyngrok import ngrok

# Inicializa o modelo LangChain
template = """Pergunta: {question}

Resposta: Vamos pensar passo a passo."""

prompt = ChatPromptTemplate.from_template(template)
model = OllamaLLM(model="llama3.1")
chain = prompt | model

# Define a função XML-RPC
def gerar_resposta(prompt_usuario):
    resposta = chain.invoke({"question": prompt_usuario})
    return resposta  # Retorna a resposta gerada

# Cria o servidor XML-RPC
class ManipuladorDeRequisicoes(SimpleXMLRPCRequestHandler):
    rpc_paths = ("/RPC2",)

server = SimpleXMLRPCServer(("0.0.0.0", 1335), requestHandler=ManipuladorDeRequisicoes, allow_none=True)
server.register_function(gerar_resposta, "generate_response")

# Inicia o túnel ngrok
url_publica = ngrok.connect(1335).public_url
print("URL Pública:", url_publica)

print("Servidor XML-RPC em execução...")
server.serve_forever()


**VII. Fazer uma requisição na máquina local**
> Ao rodar a última célula, uma URL pública será disponibilizada, copie essa URL e cole na variável do código abaixo.

> Por fim, copie o código se desejar e rode em sua máquina local, caso ocorra um erro de conexão verifique se a célula VI está em execução juntamente com o servidor XML-RPC.

In [ ]:
import xmlrpc.client

# URL utilizada para conexão do servidor RPC
url_servidor = "https://XXXXXX.ngrok-free.app" # Coloque sua URL aqui.

# Realiza a conexão com o serivor RPC pela URL.
proxy = xmlrpc.client.ServerProxy(url_servidor)

# Envia uma pergunta e recebe a resposta.
pergunta = "Qual é a capital da França?"
resposta = proxy.generate_response(pergunta)

#Exibe a resposta gerada pela IA.
print("Resposta da IA:", resposta)
